In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.providers.basic_provider import BasicSimulator
import math

# =========================================================================
# BB84 Quantum Key Distribution — With Attacker (Eve)
# =========================================================================
#
# This notebook simulates BB84 in the presence of an intercept-resend
# attacker (Eve).  Eve intercepts every qubit that Alice sends, measures
# it in a randomly chosen basis, and resends a fresh qubit to Bob in the
# state she measured.  This is the simplest possible attack.
#
# Agents:
#   ALICE  — prepares and sends qubits
#   EVE    — intercepts, measures, resends (the attacker)
#   BOB    — receives (Eve's replacement) qubits and measures
#
# Detection mechanism:
#   Because the No-Cloning Theorem prevents Eve from copying a qubit
#   without disturbing it (Lecture 3b, slide 23), her interception
#   introduces errors at matching-basis positions roughly 25% of the time.
#   Alice and Bob sacrifice a subset of their shared key bits to check for
#   errors; if the error rate exceeds a threshold, they abort.

simulator = BasicSimulator()

## Background: Eve's Attack and Why It Is Detectable

### The intercept-resend attack

Eve sits on the quantum channel between Alice and Bob.  For each qubit:

1. Eve **intercepts** the qubit and measures it in a basis she chooses randomly (standard or diagonal).
2. Eve **resends** a fresh qubit to Bob prepared in the state she measured.

### Why this introduces errors

The **No-Cloning Theorem** means Eve cannot copy the qubit and resend the original.  When Eve guesses Alice's basis **incorrectly** (probability ½) and Bob later uses the **same** basis as Alice, Eve's resent qubit is in a superposition from Bob's perspective, so Bob gets the wrong result with probability ½.

Overall error rate at matching positions:

$$P(\text{error}) = P(\text{Eve wrong basis}) \times P(\text{Bob gets wrong bit} \mid \text{Eve wrong}) = \frac{1}{2} \times \frac{1}{2} = \frac{1}{4} = 25\%$$

### Detection

Alice and Bob publicly compare a **sample** of their key bits.  With 0% legitimate error expected (no noise), any error signals eavesdropping.  By choosing a large enough sample, they can detect Eve with arbitrarily high probability and **abort** the protocol.

In [9]:
# =========================================================================
# QUANTUM RANDOMNESS (same as Plain notebook)
# =========================================================================
# All random choices — for Alice, Eve, and Bob — are made by measuring
# |+> = H|0>.  Python's random module is NOT used.

def quantum_random_bit():
    """Return a genuinely random bit by measuring |+>."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    return int(list(job.result().get_counts().keys())[0])

def quantum_random_choice(n):
    """Uniformly random index in 0..n-1 using quantum bits."""
    bits_needed = math.ceil(math.log2(n))
    while True:
        val = int(''.join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
        if val < n:
            return val

BASES = ['s', 'd']   # 's' = standard (Z), 'd' = diagonal (X)

print("Quantum random sample:", [quantum_random_bit() for _ in range(10)])

Quantum random sample: [0, 0, 1, 0, 1, 1, 0, 0, 1, 0]


In [ ]:
# =========================================================================
# ALICE: qubit encoding
# =========================================================================

def alice_encode(bit, basis):
    """ALICE — Encode `bit` in `basis` as a single-qubit circuit.

    Standard basis:  0 -> |0>,  1 -> |1>
    Diagonal basis:  0 -> |+>,  1 -> |->
    """
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 'd':
        qc.h(0)
    return qc

# =========================================================================
# BOB: qubit measurement
# =========================================================================

def bob_measure(qubit_circuit, basis):
    """BOB — Measure the qubit in `basis`.

    Standard: measure directly.
    Diagonal: apply H first (converts |+>->|0>, |->->|1>), then measure.
    """
    qc = qubit_circuit.copy()
    qc.add_register(ClassicalRegister(1, 'c'))
    if basis == 'd':
        qc.h(0)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    return int(list(job.result().get_counts().keys())[0])

print("Alice encode / Bob measure primitives defined.")

Alice encode / Bob measure primitives defined.


In [ ]:
# =========================================================================
# EVE: intercept-resend attack 
# =========================================================================
#
# Eve intercepts Alice's qubit from the channel.
# She measures it in a randomly chosen basis, then constructs a fresh
# qubit in the state she observed and forwards it to Bob.
#
# Crucially:
#   - The No-Cloning Theorem prevents Eve from sending the ORIGINAL qubit
#     to Bob; she must create a replacement.
#   - If Eve's basis differs from Alice's, the replacement qubit is wrong
#     and this will cause errors when Bob later measures in Alice's basis.

def eve_intercept_resend(qubit_circuit):
    """EVE — Measure the intercepted qubit in a random basis and resend.

    Parameters
    ----------
    qubit_circuit : QuantumCircuit
        The qubit Alice prepared (1 qubit, no classical registers).

    Returns
    -------
    eve_basis  : str  — basis Eve chose ('s' or 'd')
    eve_result : int  — bit Eve measured (0 or 1)
    replacement: QuantumCircuit — fresh qubit Eve sends to Bob
    """
    # --- EVE chooses her measurement basis using quantum randomness ---
    eve_basis = BASES[quantum_random_choice(2)]

    # --- EVE measures the intercepted qubit ---
    qc_intercept = qubit_circuit.copy()
    qc_intercept.add_register(ClassicalRegister(1, 'e'))
    if eve_basis == 'd':
        qc_intercept.h(0)     # rotate to diagonal basis before measuring
    qc_intercept.measure(0, 0)
    job = simulator.run(transpile(qc_intercept, simulator), shots=1)
    eve_result = int(list(job.result().get_counts().keys())[0])

    # --- EVE resends a fresh qubit encoding her measurement outcome ---
    # She prepares |eve_result> in eve_basis — the best she can do.
    replacement = alice_encode(eve_result, eve_basis)

    return eve_basis, eve_result, replacement

print("Eve's intercept-resend attack defined.")

Eve's intercept-resend attack defined.


In [12]:
# =========================================================================
# FULL BB84 PROTOCOL — WITH EVE
# =========================================================================

def run_bb84_attacked(n_qubits=30, sample_fraction=0.5, error_threshold=0.10):
    """
    Simulate BB84 with Eve performing an intercept-resend attack on every qubit.

    Parameters
    ----------
    n_qubits        : int   — number of qubits Alice sends
    sample_fraction : float — fraction of key bits sacrificed for error checking
    error_threshold : float — if error rate > this, protocol is aborted

    Returns
    -------
    attack_detected : bool
    """

    # ------------------------------------------------------------------
    # === ALICE: preparation ===
    # ------------------------------------------------------------------
    alice_bits  = [quantum_random_bit()            for _ in range(n_qubits)]
    alice_bases = [BASES[quantum_random_choice(2)] for _ in range(n_qubits)]
    sent_qubits = [alice_encode(b, bs) for b, bs in zip(alice_bits, alice_bases)]

    # ------------------------------------------------------------------
    # === EVE: intercept every qubit on the channel, resend replacement ===
    # ------------------------------------------------------------------
    eve_bases   = []
    eve_results = []
    intercepted_qubits = []

    for qc in sent_qubits:
        e_basis, e_result, replacement = eve_intercept_resend(qc)
        eve_bases.append(e_basis)
        eve_results.append(e_result)
        intercepted_qubits.append(replacement)  # Bob receives EVE's replacements

    # ------------------------------------------------------------------
    # === BOB: measures Eve's replacement qubits ===
    # ------------------------------------------------------------------
    bob_bases   = [BASES[quantum_random_choice(2)] for _ in range(n_qubits)]
    bob_results = [bob_measure(qc, bs) for qc, bs in zip(intercepted_qubits, bob_bases)]

    # ------------------------------------------------------------------
    # === PUBLIC DISCUSSION: basis reconciliation ===
    # ------------------------------------------------------------------
    matching = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_sifted = [alice_bits[i]   for i in matching]
    bob_sifted   = [bob_results[i]  for i in matching]

    # ------------------------------------------------------------------
    # === ERROR CHECKING: sacrifice a sample of the sifted key ===
    # ------------------------------------------------------------------
    # Alice and Bob publicly compare `sample_fraction` of the sifted bits.
    # In a perfect channel with no attacker, these must agree 100%.
    # Eve's interception causes ~25% errors, which are easily detected.

    n_sample = max(1, int(len(matching) * sample_fraction))
    # Use quantum randomness to pick which positions to sacrifice
    # Actually, use quantum choice for indices:
    all_idx = list(range(len(matching)))
    sample_idx = []
    remaining_idx = list(all_idx)
    while len(sample_idx) < n_sample and remaining_idx:
        pick = quantum_random_choice(len(remaining_idx))
        sample_idx.append(remaining_idx.pop(pick))

    sample_errors = sum(
        1 for i in sample_idx if alice_sifted[i] != bob_sifted[i]
    )
    error_rate = sample_errors / n_sample if n_sample > 0 else 0.0

    # Remaining positions (not revealed) form the candidate key
    key_idx     = [i for i in all_idx if i not in sample_idx]
    alice_key   = [alice_sifted[i] for i in key_idx]
    bob_key     = [bob_sifted[i]   for i in key_idx]

    # ------------------------------------------------------------------
    # Print trace
    # ------------------------------------------------------------------
    print(f"BB84 with Eve — {n_qubits} qubits\n")
    hdr = f"{'i':>3}  {'A bit':>6}  {'A bas':>6}  {'E bas':>6}  {'E bit':>6}  {'B bas':>6}  {'B res':>6}  {'Match':>6}  {'Error':>6}"
    print(hdr)
    print("-" * len(hdr))
    for i in range(n_qubits):
        matched = alice_bases[i] == bob_bases[i]
        err = matched and (alice_bits[i] != bob_results[i])
        print(
            f"{i:>3}  {alice_bits[i]:>6}  {alice_bases[i]:>6}  "
            f"{eve_bases[i]:>6}  {eve_results[i]:>6}  "
            f"{bob_bases[i]:>6}  {bob_results[i]:>6}  "
            f"{'YES' if matched else '---':>6}  {'ERR' if err else '   ':>6}"
        )

    print()
    print(f"Sifted key positions : {matching}")
    print(f"Sample positions     : {[matching[i] for i in sample_idx]}  ({n_sample} bits)")
    print(f"Sample errors        : {sample_errors}/{n_sample}  ({error_rate:.0%})")
    print(f"Error threshold      : {error_threshold:.0%}")
    print()

    attack_detected = error_rate > error_threshold

    if attack_detected:
        print("*** ATTACK DETECTED ***")
        print(f"Error rate {error_rate:.0%} exceeds threshold {error_threshold:.0%}.")
        print("Alice and Bob ABORT — the key is discarded.")
    else:
        print("No attack detected (error rate within threshold).")
        print(f"Remaining key — Alice: {''.join(map(str, alice_key))}")
        print(f"Remaining key — Bob  : {''.join(map(str, bob_key))}")

    return attack_detected, error_rate, alice_key, bob_key


detected, rate, ak, bk = run_bb84_attacked(n_qubits=30)

BB84 with Eve — 30 qubits

  i   A bit   A bas   E bas   E bit   B bas   B res   Match   Error
-------------------------------------------------------------------
  0       1       s       s       1       d       0     ---        
  1       0       s       s       0       s       0     YES        
  2       0       s       s       0       s       0     YES        
  3       0       s       s       0       d       0     ---        
  4       1       s       d       0       s       1     YES        
  5       1       d       d       1       s       1     ---        
  6       1       d       d       1       s       0     ---        
  7       1       d       s       1       d       0     YES     ERR
  8       0       s       d       1       d       1     ---        
  9       0       s       s       0       s       0     YES        
 10       1       d       s       0       s       0     ---        
 11       0       s       s       0       s       0     YES        
 12       0       d  

In [13]:
# =========================================================================
# STATISTICAL ANALYSIS: detection rate over many trials
# =========================================================================
# Run the attacked protocol many times and measure how reliably Eve is
# caught.  Theory predicts ~25% error rate; with a 10% threshold and a
# reasonable sample size, detection should be nearly certain.

N_TRIALS    = 20       # number of independent protocol runs
N_QUBITS    = 40       # qubits per run (more -> larger sample -> more reliable)
SAMPLE_FRAC = 0.5      # fraction of sifted bits used for error checking
THRESHOLD   = 0.10     # 10% error rate threshold

detections = 0
error_rates = []

print(f"Running {N_TRIALS} attacked trials ({N_QUBITS} qubits each, "
      f"sample={SAMPLE_FRAC:.0%}, threshold={THRESHOLD:.0%})\n")

for trial in range(N_TRIALS):
    det, rate, _, _ = run_bb84_attacked(
        n_qubits=N_QUBITS,
        sample_fraction=SAMPLE_FRAC,
        error_threshold=THRESHOLD
    )
    detections += int(det)
    error_rates.append(rate)
    print(f"  Trial {trial+1:>2}: error rate = {rate:.0%}  {'DETECTED' if det else 'missed'}")

avg_error = sum(error_rates) / len(error_rates)
print()
print("=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
print(f"Attack detected   : {detections}/{N_TRIALS} trials ({detections/N_TRIALS:.0%})")
print(f"Average error rate: {avg_error:.1%}  (theory: 25%)")
print()
print("Conclusion: Eve's intercept-resend attack introduces a ~25% error")
print("rate at sifted positions.  With a 10% threshold and adequate sample")
print("size, Alice and Bob can detect her presence with high probability.")

Running 20 attacked trials (40 qubits each, sample=50%, threshold=10%)

BB84 with Eve — 40 qubits

  i   A bit   A bas   E bas   E bit   B bas   B res   Match   Error
-------------------------------------------------------------------
  0       0       s       d       1       s       1     YES     ERR
  1       0       d       s       0       d       0     YES        
  2       1       s       d       1       d       1     ---        
  3       0       s       s       0       s       0     YES        
  4       1       d       s       1       d       1     YES        
  5       0       d       d       0       d       0     YES        
  6       0       s       d       0       s       1     YES     ERR
  7       1       d       s       1       s       1     ---        
  8       1       d       s       1       s       1     ---        
  9       1       d       s       0       s       0     ---        
 10       0       d       s       0       d       1     YES     ERR
 11       1      

In [ ]:
# =========================================================================
# EXTENSION: PARTIAL ATTACK
# =========================================================================
# Eve doesn't have to intercept every qubit. Here we simulate a partial
# attack where Eve intercepts each qubit independently with probability
# `intercept_fraction`. The expected sifted-key error rate scales as
# approximately 0.25 * intercept_fraction in noiseless BB84.

def run_bb84_partial_attack(n_qubits=40, intercept_fraction=0.5,
                            sample_fraction=0.5, error_threshold=0.10):
    """BB84 with Eve intercepting each qubit with probability intercept_fraction."""

    # ALICE
    alice_bits  = [quantum_random_bit()            for _ in range(n_qubits)]
    alice_bases = [BASES[quantum_random_choice(2)] for _ in range(n_qubits)]
    sent_qubits = [alice_encode(b, bs) for b, bs in zip(alice_bits, alice_bases)]

    # EVE: intercept each qubit independently with probability intercept_fraction
    received_by_bob = []
    for qc in sent_qubits:
        if quantum_random_choice(1000) < int(intercept_fraction * 1000):
            _, _, replacement = eve_intercept_resend(qc)
            received_by_bob.append(replacement)
        else:
            received_by_bob.append(qc)  # qubit passes through unmodified

    # BOB
    bob_bases   = [BASES[quantum_random_choice(2)] for _ in range(n_qubits)]
    bob_results = [bob_measure(qc, bs) for qc, bs in zip(received_by_bob, bob_bases)]

    # Sifting
    matching     = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_sifted = [alice_bits[i]  for i in matching]
    bob_sifted   = [bob_results[i] for i in matching]

    # Error check
    n_sample = max(1, int(len(matching) * sample_fraction))
    sample_idx = []
    remaining = list(range(len(matching)))
    while len(sample_idx) < n_sample and remaining:
        pick = quantum_random_choice(len(remaining))
        sample_idx.append(remaining.pop(pick))

    errors = sum(1 for i in sample_idx if alice_sifted[i] != bob_sifted[i])
    error_rate = errors / n_sample if n_sample > 0 else 0.0
    detected = error_rate > error_threshold

    return detected, error_rate


N_QUBITS = 200
N_TRIALS = 20
SAMPLE_FRAC = 0.5
THRESHOLD = 0.10
FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]

print("Partial attack — varying Eve's intercept fraction:")
print(f"  (n_qubits={N_QUBITS}, sample={SAMPLE_FRAC:.0%}, threshold={THRESHOLD:.0%})\n")
print(f"  {'Intercept %':>12}  {'Avg error rate':>15}  {'Theory':>8}  {'Detected':>12}")
print("  " + "-" * 58)

for frac in FRACTIONS:
    trial_errors = []
    n_det = 0
    for _ in range(N_TRIALS):
        det, rate = run_bb84_partial_attack(
            n_qubits=N_QUBITS,
            intercept_fraction=frac,
            sample_fraction=SAMPLE_FRAC,
            error_threshold=THRESHOLD,
        )
        trial_errors.append(rate)
        n_det += int(det)
    avg = sum(trial_errors) / len(trial_errors)
    theory = 0.25 * frac
    print(f"  {frac:>11.0%}  {avg:>15.1%}  {theory:>8.1%}  {n_det:>9}/{N_TRIALS}")

print()
print("Observation: in noiseless BB84, avg error should follow approximately error_rate ≈ 0.25 × intercept_fraction.")
print("Finite-sample randomness can still cause occasional non-monotonic rows.")


Partial attack — varying Eve's intercept fraction:
  (n_qubits=200, sample=50%, threshold=10%)

   Intercept %   Avg error rate    Theory      Detected
  ----------------------------------------------------------
           0%             0.0%      0.0%          0/20


## Why Some Trials Miss the Attack (and How We Improve Detection)

This implementation detects Eve by sampling only a subset of sifted bits and comparing Alice/Bob values against an error threshold.

A missed detection can occur when the sampled subset happens to contain fewer errors than the full sifted key, even though Eve attacked every qubit. This is a finite-sample statistical effect, not a protocol failure.

To reduce missed detections, we increase:

- `N_QUBITS` (more transmitted qubits -> larger sifted key), and
- `sample_fraction` (larger checked subset).

Both changes make the sample estimate closer to the true error rate (about 25% for intercept-resend in BB84), so detection probability increases.

This matches the assignment requirement to report attacks using a threshold on disrupted qubits, while showing the practical trade-off between key retention and detection confidence.


## Summary

This notebook demonstrates the **BB84 protocol under an intercept-resend attack**:

- **Eve** intercepts every qubit Alice sends, measures it in a random basis, and forwards a freshly prepared replacement to Bob.
- Because the **No-Cloning Theorem** prevents her from copying the qubit, she inevitably disturbs approximately **25%** of the sifted key bits whenever her basis choice differs from Alice's.
- Alice and Bob detect this by publicly **comparing a sample** of their sifted bits: an error rate above the 10% threshold triggers protocol abort.
- The statistical analysis confirms detection across multiple trials, and the partial-attack extension shows that even partial eavesdropping is detectable with a sufficient sample size.

All random choices (Alice's bits, Alice's bases, Eve's bases, Bob's bases, sample selection) use **quantum randomness** from measuring $|+\rangle$ — no classical pseudo-random generators are used.